# AFC-FullBench: full-episode training, testing, and visualization

This notebook trains and evaluates the full-episode AFC benchmark methods on the **TEP** and **FCC** datasets separately.

It uses complete extracted alarm-flood episodes only. It does **not** run online prefix evaluation, perturbations, trace repair, delayed-detection simulation, or robustness analysis.

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys
import math
import warnings

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import confusion_matrix

# ---------------------------------------------------------------------
# Locate repository root and import the local package.
# This works when the notebook is opened either from the repository root
# or from the notebooks/ directory.
# ---------------------------------------------------------------------
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]

PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / "src" / "afc_fullbench").is_dir():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate the repository root. "
        "Open this notebook from inside the afc-fullbench repository."
    )

os.chdir(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from afc_fullbench.data import AlarmDataset, load_alarm_series_dataset
from afc_fullbench.evaluation import run_cross_validation

In [ ]:
# ---------------------------------------------------------------------
# Notebook settings
# ---------------------------------------------------------------------

DATASET_CONFIGS = {
    "TEP": PROJECT_ROOT / "configs" / "tep.yaml",
    "FCC": PROJECT_ROOT / "configs" / "fcc.yaml",
}

METHOD_ORDER = [
    "WDI-1NN",
    "JAC-1NN",
    "EAC-1NN",
    "MBW-LR",
    "ACM-SVM",
    "CASIM",
]

# Set RUN_CV=False to skip training and load already existing CSV outputs
# from the configured result directories.
RUN_CV = True

# Save all figures directly in this single repository-level figure folder.
FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Standard file formats for notebook figures.
FIGURE_FORMATS = ("pdf", "png")

print(f"Project root: {PROJECT_ROOT}")
print(f"Figure folder: {FIGURE_DIR}")

## Helper functions

In [ ]:
def load_yaml(path: str | Path) -> dict:
    """Load one YAML configuration file."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Configuration file does not exist: {path}")
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def has_csv_files(root: str | Path) -> bool:
    """Return True if a dataset root exists and contains at least one CSV file."""
    root = Path(root)
    return root.exists() and any(root.rglob("*.csv"))


def ordered_methods(methods: list[str] | pd.Series) -> list[str]:
    """Sort methods according to the benchmark order while preserving unknown methods."""
    values = [str(m) for m in list(methods)]
    out = [m for m in METHOD_ORDER if m in values]
    out += sorted(m for m in values if m not in out)
    return out


def save_figure(fig: plt.Figure, base_path: str | Path, formats: tuple[str, ...] = FIGURE_FORMATS) -> None:
    """Save a figure in all configured formats."""
    base_path = Path(base_path)
    base_path.parent.mkdir(parents=True, exist_ok=True)
    for fmt in formats:
        out = base_path.with_suffix(f".{fmt}")
        if fmt.lower() == "png":
            fig.savefig(out, dpi=450, bbox_inches="tight")
        else:
            fig.savefig(out, bbox_inches="tight")
        print(f"saved: {out}")


def load_dataset_from_config(config: dict) -> AlarmDataset:
    """Load one dataset from a benchmark configuration."""
    data_cfg = config.get("data", {})
    root = PROJECT_ROOT / data_cfg.get("root", "")
    max_time_steps = data_cfg.get("max_time_steps", None)
    return load_alarm_series_dataset(root, max_time_steps=max_time_steps)


def dataset_overview(dataset: AlarmDataset, dataset_label: str) -> pd.DataFrame:
    """Return compact dataset metadata."""
    counts = pd.Series(dataset.y).value_counts().sort_index()
    min_class_count = int(counts.min())

    return pd.DataFrame(
        [
            {
                "dataset": dataset_label,
                "n_episodes": int(dataset.X.shape[0]),
                "n_classes": int(len(dataset.class_names)),
                "n_alarm_tags": int(dataset.X.shape[1]),
                "n_time_steps": int(dataset.X.shape[2]),
                "min_episodes_per_class": min_class_count,
            }
        ]
    )


def class_count_table(dataset: AlarmDataset) -> pd.DataFrame:
    """Return class counts."""
    counts = pd.Series(dataset.y).value_counts().sort_index()
    return pd.DataFrame(
        {
            "class_id": counts.index.astype(int),
            "class_name": [dataset.class_names[i] for i in counts.index.astype(int)],
            "n_episodes": counts.to_numpy(dtype=int),
        }
    )


def effective_cv_config(config: dict, dataset: AlarmDataset) -> dict:
    """Return a safe stratified k-fold configuration for the given dataset."""
    cv_cfg = dict(config.get("cv", {}))
    requested_splits = int(cv_cfg.get("n_splits", 5))
    min_class_count = int(pd.Series(dataset.y).value_counts().min())

    if requested_splits > min_class_count:
        warnings.warn(
            f"Requested n_splits={requested_splits}, but the smallest class has only "
            f"{min_class_count} episodes. Using n_splits={min_class_count}."
        )
        cv_cfg["n_splits"] = min_class_count
    else:
        cv_cfg["n_splits"] = requested_splits

    cv_cfg["shuffle"] = bool(cv_cfg.get("shuffle", True))
    cv_cfg["random_state"] = int(cv_cfg.get("random_state", 42))
    return cv_cfg


def load_existing_outputs(output_dir: str | Path) -> dict[str, pd.DataFrame]:
    """Load existing benchmark CSV outputs."""
    output_dir = Path(output_dir)
    required = {
        "fold_metrics": "fold_metrics.csv",
        "predictions": "predictions.csv",
        "confusion_matrices": "confusion_matrices.csv",
        "summary": "summary.csv",
    }
    missing = [name for name in required.values() if not (output_dir / name).exists()]
    if missing:
        raise FileNotFoundError(
            f"Cannot load existing outputs from {output_dir}; missing files: {missing}"
        )
    return {key: pd.read_csv(output_dir / filename) for key, filename in required.items()}


def sort_summary(summary: pd.DataFrame) -> pd.DataFrame:
    """Sort a summary table by the benchmark method order."""
    summary = summary.copy()
    order = {method: i for i, method in enumerate(METHOD_ORDER)}
    summary["_order"] = summary["method"].map(order).fillna(10_000).astype(int)
    summary = summary.sort_values(["_order", "method"]).drop(columns="_order")
    return summary.reset_index(drop=True)

In [ ]:
def plot_class_distribution(dataset: AlarmDataset, dataset_label: str) -> plt.Figure:
    """Plot the class distribution of one dataset."""
    table = class_count_table(dataset)

    fig_width = max(5.6, 0.38 * len(table))
    fig, ax = plt.subplots(figsize=(fig_width, 3.2))

    x = np.arange(len(table))
    ax.bar(x, table["n_episodes"].to_numpy())
    ax.set_xticks(x)
    ax.set_xticklabels(table["class_name"].astype(str), rotation=45, ha="right")
    ax.set_ylabel("Number of episodes")
    ax.set_xlabel("Alarm-flood class")
    ax.set_title(f"{dataset_label}: class distribution")
    ax.grid(axis="y", linewidth=0.4, alpha=0.4)
    fig.tight_layout()

    save_figure(fig, FIGURE_DIR / f"{dataset_label.lower()}_class_distribution")
    return fig


def plot_metric_summary(summary: pd.DataFrame, dataset_label: str) -> plt.Figure:
    """Plot mean cross-validation metrics by method."""
    summary = sort_summary(summary)

    metric_specs = [
        ("accuracy", "Accuracy"),
        ("balanced_accuracy", "Balanced accuracy"),
        ("macro_f1", "Macro F1"),
        ("weighted_f1", "Weighted F1"),
    ]

    methods = summary["method"].astype(str).tolist()
    x = np.arange(len(methods))
    width = 0.18

    fig_width = max(7.2, 0.85 * len(methods))
    fig, ax = plt.subplots(figsize=(fig_width, 4.0))

    offsets = (np.arange(len(metric_specs)) - (len(metric_specs) - 1) / 2.0) * width
    for offset, (metric, label) in zip(offsets, metric_specs):
        mean_col = f"{metric}_mean"
        std_col = f"{metric}_std"
        if mean_col not in summary.columns:
            continue

        means = summary[mean_col].astype(float).to_numpy()
        yerr = summary[std_col].astype(float).to_numpy() if std_col in summary.columns else None

        ax.bar(x + offset, means, width=width, label=label)
        if yerr is not None:
            ax.errorbar(x + offset, means, yerr=yerr, fmt="none", capsize=2, linewidth=0.8)

    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel("Cross-validation score")
    ax.set_xlabel("AFC method")
    ax.set_title(f"{dataset_label}: full-episode classification performance")
    ax.set_xticks(x)
    ax.set_xticklabels(methods, rotation=30, ha="right")
    ax.grid(axis="y", linewidth=0.4, alpha=0.4)
    ax.legend(loc="lower right", frameon=True)
    fig.tight_layout()

    save_figure(fig, FIGURE_DIR / f"{dataset_label.lower()}_metric_summary")
    return fig


def plot_confusion_matrix_grid(
    predictions: pd.DataFrame,
    dataset: AlarmDataset,
    dataset_label: str,
    *,
    normalize: str | None = "true",
) -> plt.Figure:
    """Plot pooled confusion matrices for all methods of one dataset."""
    methods = ordered_methods(predictions["method"].unique())
    class_ids = np.arange(len(dataset.class_names), dtype=int)
    class_labels = [str(c) for c in dataset.class_names]

    n_methods = len(methods)
    n_cols = 3
    n_rows = math.ceil(n_methods / n_cols)

    fig_width = 3.2 * n_cols
    fig_height = 3.0 * n_rows
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(fig_width, fig_height),
        squeeze=False,
        constrained_layout=True,
    )

    ims = []
    annotate = len(class_labels) <= 8

    for idx, method in enumerate(methods):
        row, col = divmod(idx, n_cols)
        ax = axes[row, col]
        sub = predictions[predictions["method"].astype(str) == method]

        y_true = sub["y_true"].astype(int).to_numpy()
        y_pred = sub["y_pred"].astype(int).to_numpy()

        cm = confusion_matrix(y_true, y_pred, labels=class_ids, normalize=normalize)
        im = ax.imshow(cm, vmin=0.0 if normalize else None, vmax=1.0 if normalize else None)
        ims.append(im)

        ax.set_title(method)
        ax.set_xticks(np.arange(len(class_labels)))
        ax.set_yticks(np.arange(len(class_labels)))
        ax.set_xticklabels(class_labels, rotation=45, ha="right", fontsize=7)
        ax.set_yticklabels(class_labels, fontsize=7)
        ax.set_xlabel("Predicted")
        if col == 0:
            ax.set_ylabel("True")

        if annotate:
            for i in range(cm.shape[0]):
                for j in range(cm.shape[1]):
                    value = cm[i, j]
                    label = f"{value:.2f}" if normalize else f"{int(value)}"
                    ax.text(j, i, label, ha="center", va="center", fontsize=6)

    for idx in range(n_methods, n_rows * n_cols):
        row, col = divmod(idx, n_cols)
        axes[row, col].axis("off")

    cbar = fig.colorbar(ims[0], ax=axes.ravel().tolist(), fraction=0.025, pad=0.015)
    cbar.set_label("Normalized count" if normalize else "Count")

    fig.suptitle(f"{dataset_label}: pooled confusion matrices", y=1.02)
    save_figure(fig, FIGURE_DIR / f"{dataset_label.lower()}_confusion_matrices")
    return fig

In [ ]:
def run_dataset_pipeline(dataset_label: str, config_path: str | Path) -> dict | None:
    """Run full-episode training/testing and visualizations for one dataset."""
    config_path = Path(config_path)
    config = load_yaml(config_path)

    data_root = PROJECT_ROOT / config.get("data", {}).get("root", "")
    output_dir = PROJECT_ROOT / config.get("output_dir", f"results/{dataset_label.lower()}_full_episode")
    output_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 80)
    print(f"{dataset_label}: {config_path}")
    print(f"Data root: {data_root}")
    print(f"Output directory: {output_dir}")
    print("=" * 80)

    if not has_csv_files(data_root):
        print(
            f"No CSV files found under {data_root}. "
            "Add the dataset first, then rerun this cell."
        )
        return None

    dataset = load_dataset_from_config(config)

    overview = dataset_overview(dataset, dataset_label)
    counts = class_count_table(dataset)

    print("Dataset overview")
    display(overview)

    print("Class counts")
    display(counts)

    cv_cfg = effective_cv_config(config, dataset)

    if RUN_CV:
        outputs = run_cross_validation(
            dataset,
            model_configs=config.get("models", []),
            n_splits=int(cv_cfg["n_splits"]),
            shuffle=bool(cv_cfg["shuffle"]),
            random_state=int(cv_cfg["random_state"]),
            output_dir=output_dir,
        )
    else:
        outputs = load_existing_outputs(output_dir)

    summary = sort_summary(outputs["summary"])
    print("Cross-validation summary")
    display(summary)

    fig_class = plot_class_distribution(dataset, dataset_label)
    display(fig_class)
    plt.close(fig_class)

    fig_metrics = plot_metric_summary(summary, dataset_label)
    display(fig_metrics)
    plt.close(fig_metrics)

    fig_cm = plot_confusion_matrix_grid(outputs["predictions"], dataset, dataset_label)
    display(fig_cm)
    plt.close(fig_cm)

    return {
        "dataset": dataset,
        "config": config,
        "output_dir": output_dir,
        "overview": overview,
        "class_counts": counts,
        "outputs": outputs,
        "summary": summary,
    }

## TEP full-episode benchmark

In [ ]:
tep_result = run_dataset_pipeline("TEP", DATASET_CONFIGS["TEP"])

## FCC full-episode benchmark

In [ ]:
fcc_result = run_dataset_pipeline("FCC", DATASET_CONFIGS["FCC"])

## Compact result tables

The following cell exports a compact combined summary table for the datasets that were successfully evaluated.

In [ ]:
summaries = []

for dataset_label, result in [("TEP", tep_result), ("FCC", fcc_result)]:
    if result is None:
        continue

    summary = result["summary"].copy()
    summary.insert(0, "dataset", dataset_label)
    summaries.append(summary)

if summaries:
    combined_summary = pd.concat(summaries, ignore_index=True)
    out = PROJECT_ROOT / "results" / "full_episode_combined_summary.csv"
    out.parent.mkdir(parents=True, exist_ok=True)
    combined_summary.to_csv(out, index=False)

    print(f"Saved combined summary to: {out}")
    display(combined_summary)
else:
    print("No dataset results available yet.")